In [ ]:
!pip install catboost

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [ ]:
# Step 1: Load the Base Models (DenseNet169, Xception, EfficientNetB4, DenseNet121 for Fruit)
fruit_model_1_path = '/content/drive/MyDrive/Capstone Models/DenseNet169_fruit.keras'
fruit_model_2_path = '/content/drive/MyDrive/Capstone Models/Xception_fruit.keras'
fruit_model_3_path = '/content/drive/MyDrive/Capstone Models/EfficientNetB4_fruit.keras'
fruit_model_4_path = '/content/drive/MyDrive/Capstone Models/DenseNet121_fruit.keras'

fruit_model_1 = load_model(fruit_model_1_path)
fruit_model_2 = load_model(fruit_model_2_path)
fruit_model_3 = load_model(fruit_model_3_path)
fruit_model_4 = load_model(fruit_model_4_path)

In [ ]:
# Step 2: Load the Dataset (Fruit Test Dataset)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

test_dir = "/content/drive/MyDrive/Capstone Dataset/Guava_Fruit_Dieases/Test"

# Define fruit classes manually
fruit_classes = ['Anthracnose', 'Healthy', 'Scab', 'Styler end root']

# Load the test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 880 files belonging to 4 classes.


In [ ]:
# Step 3: Generate Predictions from Base Models
y_prob_fruit_model_1 = fruit_model_1.predict(test_ds)  # Predictions from DenseNet169
y_prob_fruit_model_2 = fruit_model_2.predict(test_ds)  # Predictions from Xception
y_prob_fruit_model_3 = fruit_model_3.predict(test_ds)  # Predictions from EfficientNetB4
y_prob_fruit_model_4 = fruit_model_4.predict(test_ds)  # Predictions from DenseNet121

28/28 ━━━━━━━━━━━━━━━━━━━━ 146s 5s/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 204ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 15s 339ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 261ms/step


In [ ]:
# Step 4: Stack the Predictions (Create Meta-Features)
X_meta_fruit = np.column_stack((
    y_prob_fruit_model_1,
    y_prob_fruit_model_2,
    y_prob_fruit_model_3,
    y_prob_fruit_model_4
))

# True labels for the test set (adjust this to match your dataset)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

# Step 5: Define and Train Meta-Models (Each Section)

In [ ]:
## 1. Logistic Regression
print("\nTraining Logistic Regression...")
meta_model_lr = LogisticRegression()
meta_model_lr.fit(X_meta_fruit, y_true)
y_meta_pred_lr = meta_model_lr.predict(X_meta_fruit)
print(f"Logistic Regression Accuracy: {accuracy_score(y_true, y_meta_pred_lr):.4f}")
print(classification_report(y_true, y_meta_pred_lr, target_names=fruit_classes))


Training Logistic Regression...
Logistic Regression Accuracy: 0.8000
                 precision    recall  f1-score   support

    Anthracnose       0.75      0.59      0.66       220
        Healthy       0.80      0.94      0.86       220
           Scab       0.77      0.79      0.78       220
Styler end root       0.87      0.89      0.88       220

       accuracy                           0.80       880
      macro avg       0.80      0.80      0.79       880
   weighted avg       0.80      0.80      0.79       880



In [ ]:
## 2. Random Forest
print("\nTraining Random Forest...")
meta_model_rf = RandomForestClassifier()
meta_model_rf.fit(X_meta_fruit, y_true)
y_meta_pred_rf = meta_model_rf.predict(X_meta_fruit)
print(f"Random Forest Accuracy: {accuracy_score(y_true, y_meta_pred_rf):.4f}")
print(classification_report(y_true, y_meta_pred_rf, target_names=fruit_classes))


Training Random Forest...
Random Forest Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 3. Gradient Boosting
print("\nTraining Gradient Boosting...")
meta_model_gb = GradientBoostingClassifier()
meta_model_gb.fit(X_meta_fruit, y_true)
y_meta_pred_gb = meta_model_gb.predict(X_meta_fruit)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_true, y_meta_pred_gb):.4f}")
print(classification_report(y_true, y_meta_pred_gb, target_names=fruit_classes))


Training Gradient Boosting...
Gradient Boosting Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 4. Support Vector Classifier (SVC)
print("\nTraining Support Vector Classifier (SVC)...")
meta_model_svc = SVC()
meta_model_svc.fit(X_meta_fruit, y_true)
y_meta_pred_svc = meta_model_svc.predict(X_meta_fruit)
print(f"SVC Accuracy: {accuracy_score(y_true, y_meta_pred_svc):.4f}")
print(classification_report(y_true, y_meta_pred_svc, target_names=fruit_classes))


Training Support Vector Classifier (SVC)...
SVC Accuracy: 0.8705
                 precision    recall  f1-score   support

    Anthracnose       0.85      0.84      0.84       220
        Healthy       0.80      0.95      0.87       220
           Scab       0.89      0.82      0.85       220
Styler end root       0.97      0.88      0.92       220

       accuracy                           0.87       880
      macro avg       0.88      0.87      0.87       880
   weighted avg       0.88      0.87      0.87       880



In [ ]:
## 5. Multi-layer Perceptron (MLP)
print("\nTraining Multi-layer Perceptron (MLP)...")
meta_model_mlp = MLPClassifier()
meta_model_mlp.fit(X_meta_fruit, y_true)
y_meta_pred_mlp = meta_model_mlp.predict(X_meta_fruit)
print(f"MLP Accuracy: {accuracy_score(y_true, y_meta_pred_mlp):.4f}")
print(classification_report(y_true, y_meta_pred_mlp, target_names=fruit_classes))


Training Multi-layer Perceptron (MLP)...
MLP Accuracy: 0.9011
                 precision    recall  f1-score   support

    Anthracnose       0.88      0.90      0.89       220
        Healthy       0.86      0.95      0.90       220
           Scab       0.91      0.84      0.87       220
Styler end root       0.97      0.91      0.94       220

       accuracy                           0.90       880
      macro avg       0.90      0.90      0.90       880
   weighted avg       0.90      0.90      0.90       880



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
## 6. AdaBoost Classifier
print("\nTraining AdaBoost Classifier...")
meta_model_ada = AdaBoostClassifier()
meta_model_ada.fit(X_meta_fruit, y_true)
y_meta_pred_ada = meta_model_ada.predict(X_meta_fruit)
print(f"AdaBoost Accuracy: {accuracy_score(y_true, y_meta_pred_ada):.4f}")
print(classification_report(y_true, y_meta_pred_ada, target_names=fruit_classes))


Training AdaBoost Classifier...
AdaBoost Accuracy: 0.8250
                 precision    recall  f1-score   support

    Anthracnose       0.75      0.72      0.73       220
        Healthy       0.83      0.88      0.86       220
           Scab       0.83      0.86      0.85       220
Styler end root       0.89      0.84      0.86       220

       accuracy                           0.82       880
      macro avg       0.82      0.82      0.82       880
   weighted avg       0.82      0.82      0.82       880



In [ ]:
## 7. Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
meta_model_dt = DecisionTreeClassifier()
meta_model_dt.fit(X_meta_fruit, y_true)
y_meta_pred_dt = meta_model_dt.predict(X_meta_fruit)
print(f"Decision Tree Accuracy: {accuracy_score(y_true, y_meta_pred_dt):.4f}")
print(classification_report(y_true, y_meta_pred_dt, target_names=fruit_classes))


Training Decision Tree Classifier...
Decision Tree Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 8. K-Nearest Neighbors (KNN)
print("\nTraining K-Nearest Neighbors (KNN)...")
meta_model_knn = KNeighborsClassifier()
meta_model_knn.fit(X_meta_fruit, y_true)
y_meta_pred_knn = meta_model_knn.predict(X_meta_fruit)
print(f"KNN Accuracy: {accuracy_score(y_true, y_meta_pred_knn):.4f}")
print(classification_report(y_true, y_meta_pred_knn, target_names=fruit_classes))


Training K-Nearest Neighbors (KNN)...
KNN Accuracy: 0.9568
                 precision    recall  f1-score   support

    Anthracnose       0.97      0.94      0.95       220
        Healthy       0.95      0.96      0.96       220
           Scab       0.94      0.94      0.94       220
Styler end root       0.97      0.99      0.98       220

       accuracy                           0.96       880
      macro avg       0.96      0.96      0.96       880
   weighted avg       0.96      0.96      0.96       880



In [ ]:
## 9. Gaussian Naive Bayes (GNB)
print("\nTraining Gaussian Naive Bayes (GNB)...")
meta_model_gnb = GaussianNB()
meta_model_gnb.fit(X_meta_fruit, y_true)
y_meta_pred_gnb = meta_model_gnb.predict(X_meta_fruit)
print(f"GNB Accuracy: {accuracy_score(y_true, y_meta_pred_gnb):.4f}")
print(classification_report(y_true, y_meta_pred_gnb, target_names=fruit_classes))


Training Gaussian Naive Bayes (GNB)...
GNB Accuracy: 0.7625
                 precision    recall  f1-score   support

    Anthracnose       0.75      0.44      0.56       220
        Healthy       0.77      0.93      0.84       220
           Scab       0.76      0.74      0.75       220
Styler end root       0.76      0.95      0.84       220

       accuracy                           0.76       880
      macro avg       0.76      0.76      0.75       880
   weighted avg       0.76      0.76      0.75       880



In [ ]:
## 10. Quadratic Discriminant Analysis (QDA)
print("\nTraining Quadratic Discriminant Analysis (QDA)...")
meta_model_qda = QuadraticDiscriminantAnalysis()
meta_model_qda.fit(X_meta_fruit, y_true)
y_meta_pred_qda = meta_model_qda.predict(X_meta_fruit)
print(f"QDA Accuracy: {accuracy_score(y_true, y_meta_pred_qda):.4f}")
print(classification_report(y_true, y_meta_pred_qda, target_names=fruit_classes))


Training Quadratic Discriminant Analysis (QDA)...
QDA Accuracy: 0.8250
                 precision    recall  f1-score   support

    Anthracnose       0.83      0.60      0.69       220
        Healthy       0.84      0.95      0.89       220
           Scab       0.74      0.81      0.78       220
Styler end root       0.89      0.95      0.92       220

       accuracy                           0.82       880
      macro avg       0.83      0.82      0.82       880
   weighted avg       0.83      0.82      0.82       880



/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


In [ ]:
## 11. XGBoost Classifier
print("\nTraining XGBoost Classifier...")
meta_model_xgb = XGBClassifier()
meta_model_xgb.fit(X_meta_fruit, y_true)
y_meta_pred_xgb = meta_model_xgb.predict(X_meta_fruit)
print(f"XGBoost Accuracy: {accuracy_score(y_true, y_meta_pred_xgb):.4f}")
print(classification_report(y_true, y_meta_pred_xgb, target_names=fruit_classes))


Training XGBoost Classifier...
XGBoost Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 12. CatBoost Classifier
print("\nTraining CatBoost Classifier...")
meta_model_cat = CatBoostClassifier(learning_rate=0.1, iterations=1000, depth=6)
meta_model_cat.fit(X_meta_fruit, y_true)
y_meta_pred_cat = meta_model_cat.predict(X_meta_fruit)
print(f"CatBoost Accuracy: {accuracy_score(y_true, y_meta_pred_cat):.4f}")
print(classification_report(y_true, y_meta_pred_cat, target_names=fruit_classes))


Training CatBoost Classifier...
0:	learn: 1.2564890	total: 62.1ms	remaining: 1m 2s
1:	learn: 1.1488632	total: 72.2ms	remaining: 36s
2:	learn: 1.0645563	total: 81.9ms	remaining: 27.2s
3:	learn: 0.9907715	total: 91.7ms	remaining: 22.8s
4:	learn: 0.9238554	total: 102ms	remaining: 20.3s
5:	learn: 0.8631628	total: 112ms	remaining: 18.5s
6:	learn: 0.8090732	total: 122ms	remaining: 17.3s
7:	learn: 0.7627007	total: 132ms	remaining: 16.4s
8:	learn: 0.7258794	total: 142ms	remaining: 15.6s
9:	learn: 0.6861422	total: 152ms	remaining: 15.1s
10:	learn: 0.6543629	total: 162ms	remaining: 14.6s
11:	learn: 0.6244423	total: 174ms	remaining: 14.3s
12:	learn: 0.5983824	total: 187ms	remaining: 14.2s
13:	learn: 0.5745750	total: 206ms	remaining: 14.5s
14:	learn: 0.5498240	total: 226ms	remaining: 14.8s
15:	learn: 0.5291834	total: 236ms	remaining: 14.5s
16:	learn: 0.5094743	total: 247ms	remaining: 14.3s
17:	learn: 0.4913029	total: 257ms	remaining: 14s
18:	learn: 0.4733734	total: 268ms	remaining: 13.8s
19:	lear

In [ ]:
## 13. LightGBM Classifier
print("\nTraining LightGBM Classifier...")
meta_model_lgbm = LGBMClassifier()
meta_model_lgbm.fit(X_meta_fruit, y_true)
y_meta_pred_lgbm = meta_model_lgbm.predict(X_meta_fruit)
print(f"LightGBM Accuracy: {accuracy_score(y_true, y_meta_pred_lgbm):.4f}")
print(classification_report(y_true, y_meta_pred_lgbm, target_names=fruit_classes))


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000449 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 880, number of used features: 16
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
